In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

# -------------------------------------------------------------------
# 1. Configuration & Target Unity Catalog Naming Structure
# -------------------------------------------------------------------
SECRET_SCOPE = "joshua_scope"
SECRET_KEY = "joshuandegwa-evh-cs"
EVENT_HUB_NAME = "JoshuaNdegwa-evh"

catalog_name = "dbr_dev"
login_prefix = "joshuandegwa"
schema_name = f"{login_prefix}_bronze"
volume_name = "eventhub"
target_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
table_name = "opensky_flights_stream"
checkpoint_path = f"{target_volume_path}/_checkpoints/{table_name}"

# Ensure Unity Catalog schema and volume exist
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

# Retrieve connection string securely
connection_string = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# Extract namespace hostname for the Kafka-compatible endpoint
namespace_match = re.search(r'sb://([^/]+)', connection_string)
hostname = namespace_match.group(1) if namespace_match else ""
kafka_bootstrap_servers = f"{hostname}:9093"

# -------------------------------------------------------------------
# 2. Read Stream from Event Hub via Kafka endpoint (With Shaded JAAS Fix)
# -------------------------------------------------------------------
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", EVENT_HUB_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    # NOTE: 'kafkashaded.' prefix is mandatory for Databricks runtime's shaded client packaging
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";')
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "30000")
    .load()
)

# Event Hubs (Kafka endpoint) stores the payload in the 'value' column as binary
body_df = raw_stream_df.select(F.col("value").cast(StringType()).alias("json_body"))

# -------------------------------------------------------------------
# 3. Define Schema & Parse JSON Payload
# -------------------------------------------------------------------
flight_schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("baro_altitude_m", DoubleType(), True),
    StructField("on_ground", BooleanType(), True),
    StructField("velocity_mps", DoubleType(), True),
    StructField("true_track_deg", DoubleType(), True),
    StructField("api_timestamp", LongType(), True),
    StructField("ingested_at", StringType(), True)
])

parsed_stream_df = body_df.select(
    F.from_json(F.col("json_body"), flight_schema).alias("data")
).select("data.*")

# Add processing timestamp
final_stream_df = parsed_stream_df.withColumn(
    "processed_at", F.current_timestamp()
)

# -------------------------------------------------------------------
# 4. Write Stream to Unity Catalog Delta Table with Cost-Aware Trigger
# -------------------------------------------------------------------
query = (
    final_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .table(f"{catalog_name}.{schema_name}.{table_name}")
)

print(f"🚀 Streaming query started, writing to Unity Catalog: {catalog_name}.{schema_name}.{table_name}")
print(f"📁 Checkpoint location mapped to Volume path: {checkpoint_path}")

In [0]:
# import re
# from pyspark.sql import functions as F
# from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

# # -------------------------------------------------------------------
# # 1. Configuration & Target Unity Catalog Naming Structure
# # -------------------------------------------------------------------
# SECRET_SCOPE = "joshua_scope"
# SECRET_KEY = "joshuandegwa-evh-cs"
# EVENT_HUB_NAME = "JoshuaNdegwa-evh"

# catalog_name = "dbr_dev"
# login_prefix = "joshuandegwa"
# schema_name = f"{login_prefix}_bronze"
# volume_name = "eventhub"
# target_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
# table_name = "opensky_flights_stream"
# checkpoint_path = f"{target_volume_path}/_checkpoints/{table_name}"

# # Ensure Unity Catalog schema and volume exist
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

# # Retrieve connection string securely
# connection_string = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# # Extract namespace hostname for the Kafka-compatible endpoint
# namespace_match = re.search(r'sb://([^/]+)', connection_string)
# hostname = namespace_match.group(1) if namespace_match else ""
# kafka_bootstrap_servers = f"{hostname}:9093"

# # -------------------------------------------------------------------
# # 2. Read Stream from Event Hub via Kafka endpoint
# # -------------------------------------------------------------------
# raw_stream_df = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
#     .option("subscribe", EVENT_HUB_NAME)
#     .option("kafka.security.protocol", "SASL_SSL")
#     .option("kafka.sasl.mechanism", "PLAIN")
#     .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";')
#     .option("kafka.request.timeout.ms", "60000")
#     .option("kafka.session.timeout.ms", "30000")
#     .load()
# )

# # -------------------------------------------------------------------
# # 3. Enrich Stream with Bronze Metadata Natively (No UDFs)
# # -------------------------------------------------------------------
# bronze_stream_df = raw_stream_df.select(
#     F.col("value").cast("string").alias("raw_payload"),
#     F.col("offset").alias("meta_offset"),
#     F.col("partition").alias("meta_partition"),
#     F.col("timestamp").alias("meta_enqueued_timestamp"),
#     F.current_timestamp().alias("meta_processed_at")
# )

# # -------------------------------------------------------------------
# # 4. Write Stream to Bronze Delta Table with Safe Checkpointing & Cost Trigger
# # -------------------------------------------------------------------
# bronze_query = (
#     bronze_stream_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", checkpoint_path)
#     .trigger(availableNow=True)
#     .table(f"{catalog_name}.{schema_name}.{table_name}")
# )

# print(f"🚀 Bronze streaming job started, writing to Unity Catalog: {catalog_name}.{schema_name}.{table_name}")
# print(f"📁 Checkpoint location mapped to Volume path: {checkpoint_path}")